In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path

# Reproducibility
SEED = 7

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the preprocessed financial dataset
DATA_PATH = Path("data/financial_train_test.pt")

if not DATA_PATH.exists():
    DATA_PATH = Path("experiments/03_financial/data/financial_train_test.pt")

bundle = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False
)

train_data = bundle["train"]
test_data = bundle["test"]

INPUT_DIM = 2

print("Device:", DEVICE)
print("Train:", train_data.shape)
print("Test:", test_data.shape)
print("Input dimension:", INPUT_DIM)
print("Channels:", bundle["channels"])

Device: cuda
Train: torch.Size([2320, 2])
Test: torch.Size([580, 2])
Input dimension: 2
Channels: ['BTC', 'ETH']


In [3]:
def orthogonalize(W: torch.Tensor) -> torch.Tensor:
    Q, R = torch.linalg.qr(W)
    return Q


class StrictStableKoopmanOperator(nn.Module):

    def __init__(self, latent_dim: int, rho_max: float = 0.99):
        super().__init__()

        self.U_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.V_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.S_raw = nn.Parameter(
            torch.randn(latent_dim)
        )

        self.rho_max = rho_max

    def forward(self, z: torch.Tensor):

        # Orthogonal matrices
        U = orthogonalize(self.U_raw)
        V = orthogonalize(self.V_raw)

        # Constrained singular values
        Sigma = torch.sigmoid(self.S_raw) * self.rho_max

        # Construct Koopman operator
        K = U @ torch.diag(Sigma) @ V.T

        # Transform latent representations
        z_next = z @ K.T

        return z_next, K, Sigma

In [5]:
class SinCosPosEnc(nn.Module):

    def __init__(self, d_model: int, max_len: int = 10000):
        super().__init__()

        # Vị trí từ 0 đến max_len - 1
        pos = torch.arange(max_len).float().unsqueeze(1)

        # Hệ số tần số cho từng cặp chiều
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10000.0) / d_model)
        )

        # Khởi tạo positional encoding
        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        # Lưu vào model nhưng không coi là trainable parameter
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]

In [6]:
class PatchEmbed1D(nn.Module):

    def __init__(
        self,
        in_ch: int,
        d_model: int,
        patch_len: int,
        stride: int
    ):
        super().__init__()

        self.conv = nn.Conv1d(
            in_channels=in_ch,
            out_channels=d_model,
            kernel_size=patch_len,
            stride=stride
        )

    def forward(self, x):

        # (Batch, Time, Channels) -> (Batch, Channels, Time)
        x = x.permute(0, 2, 1)

        # Convolution tạo patch embeddings
        x = self.conv(x)

        # (Batch, d_model, N_patches) -> (Batch, N_patches, d_model)
        x = x.permute(0, 2, 1)

        return x

In [8]:
class PatchTST_Backbone(nn.Module):

    def __init__(
        self,
        input_dim: int,
        seq_len: int,
        patch_len: int,
        d_model: int = 48,
        num_layers: int = 3,
        num_heads: int = 4,
        dim_ff: int = 96
    ):
        super().__init__()

        # 1. Chia chuỗi thành các patch và embedding
        self.patch = PatchEmbed1D(
            in_ch=input_dim,
            d_model=d_model,
            patch_len=patch_len,
            stride=patch_len
        )

        # 2. Positional Encoding
        n_patches = int(np.ceil(seq_len / patch_len))

        self.pos = SinCosPosEnc(
            d_model=d_model,
            max_len=n_patches + 1
        )

        # 3. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # 4. Learnable CLS Token
        self.cls = nn.Parameter(
            torch.randn(1, 1, d_model)
        )

    def forward(self, x):

        # (B, seq_len, input_dim) -> (B, n_patches, d_model)
        x = self.patch(x)

        # Mở rộng CLS token cho toàn bộ batch
        cls = self.cls.expand(x.size(0), -1, -1)

        # Ghép CLS token vào đầu chuỗi
        x = torch.cat([cls, x], dim=1)

        # Positional Encoding + Transformer Encoder
        x = self.encoder(self.pos(x))

        # Lấy biểu diễn tại CLS token
        z = x[:, 0]

        return z

In [9]:
class Koopformer_PatchTST(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon,
        patch_len,
        d_model=48
    ):
        super().__init__()

        # PatchTST backbone
        self.backbone = PatchTST_Backbone(
            input_dim=input_dim,
            seq_len=seq_len,
            patch_len=patch_len,
            d_model=d_model
        )

        # Koopman operator
        self.koop = StrictStableKoopmanOperator(
            latent_dim=d_model
        )

        # Forecasting head
        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # Encode input into latent space
        z = self.backbone(x)

        # Apply Koopman operator
        z_next, K, Sigma = self.koop(z)

        # Forecast future values
        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred

In [12]:
class SeriesDecomp(nn.Module):

    def __init__(self, k: int = 3):
        super().__init__()

        self.avg = nn.AvgPool1d(
            kernel_size=k,
            stride=1,
            padding=k // 2
        )

    def forward(self, x):

        # (B, T, C) -> (B, C, T)
        x_transposed = x.transpose(1, 2)

        # Moving Average -> Trend
        trend = self.avg(x_transposed)

        # (B, C, T) -> (B, T, C)
        trend = trend.transpose(1, 2)

        # Với kernel chẵn, AvgPool có thể sinh thêm 1 time step
        if trend.size(1) != x.size(1):
            trend = trend[:, :x.size(1)]

        # Seasonal = Original - Trend
        seasonal = x - trend

        return trend, seasonal

In [13]:
class SimpleAutoformer(nn.Module):

    def __init__(
        self,
        input_len,
        horizon,
        input_dim,
        patch_len,
        d_model=48,
        num_heads=4,
        dim_ff=64,
        num_layers=3
    ):
        super().__init__()

        # 1. Series Decomposition
        self.dec = SeriesDecomp(k=patch_len)

        # 2. Seasonal branch
        self.embed = nn.Linear(input_dim, d_model)

        self.pos = SinCosPosEnc(
            d_model=d_model,
            max_len=input_len
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # 3. Forecasting heads for both components
        self.fc_seas = nn.Linear(d_model, horizon)

        self.fc_trend = nn.Linear(
            input_len * input_dim,
            horizon
        )

    def forward(self, x):

        # Separate trend and seasonal components
        trend, seas = self.dec(x)

        # Seasonal branch: Transformer Encoder
        seas = self.embed(seas)
        seas = self.pos(seas)
        seas = self.encoder(seas)

        # Aggregate seasonal representations
        seas_o = self.fc_seas(seas.mean(dim=1))

        # Trend branch: Flatten + Linear
        trend_flat = trend.reshape(trend.size(0), -1)
        trend_o = self.fc_trend(trend_flat)

        # Combine both branches
        return seas_o + trend_o

In [14]:
class Koopformer_Autoformer(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon,
        patch_len,
        d_model=48
    ):
        super().__init__()

        # Autoformer backbone
        self.backbone = SimpleAutoformer(
            input_len=seq_len,
            horizon=horizon,
            input_dim=input_dim,
            patch_len=patch_len,
            d_model=d_model
        )

        # Koopman operates in H-dimensional latent space
        self.koop = StrictStableKoopmanOperator(
            latent_dim=horizon
        )

        # Forecasting head
        self.fc = nn.Linear(
            horizon,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        z = self.backbone(x)

        z_next, K, Sigma = self.koop(z)

        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred

In [16]:
class InformerSparse(nn.Module):

    def __init__(
        self,
        input_dim,
        d_model=48,
        num_heads=4,
        dim_ff=96,
        num_layers=3,
        seq_len=120,
        patch_len=1
    ):
        super().__init__()

        # Có sử dụng Patch Embedding hay không?
        self.use_patch = patch_len > 1

        if self.use_patch:
            self.patch = PatchEmbed1D(
                in_ch=input_dim,
                d_model=d_model,
                patch_len=patch_len,
                stride=patch_len
            )

            n_tokens = int(np.ceil(seq_len / patch_len))

        else:
            self.embed = nn.Linear(
                input_dim,
                d_model
            )

            n_tokens = seq_len

        # Positional Encoding
        self.pos = SinCosPosEnc(
            d_model=d_model,
            max_len=n_tokens
        )

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):

        # 1. Input Embedding
        if self.use_patch:
            x = self.patch(x)
        else:
            x = self.embed(x)

        # 2. Positional Encoding
        x = self.pos(x)

        # 3. Transformer Encoder
        x = self.encoder(x)

        # 4. Mean Pooling across tokens
        z = x.mean(dim=1)

        return z

In [17]:
class Koopformer_Informer(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon,
        patch_len,
        d_model=48
    ):
        super().__init__()

        # Informer Backbone
        self.backbone = InformerSparse(
            input_dim=input_dim,
            d_model=d_model,
            seq_len=seq_len,
            patch_len=patch_len
        )

        # Koopman Operator
        self.koop = StrictStableKoopmanOperator(
            latent_dim=d_model
        )

        # Forecasting Head
        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # Extract latent representation
        z = self.backbone(x)

        # Apply Koopman transformation
        z_next, K, Sigma = self.koop(z)

        # Forecast BTC and ETH
        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred

In [19]:
class SimpleLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers,
        horizon
    ):
        super().__init__()

        # 1. LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        # 2. Forecasting head
        self.fc = nn.Linear(
            hidden_dim,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # h_n: (num_layers, batch_size, hidden_dim)
        _, (h_n, c_n) = self.lstm(x)

        # Final hidden state of the last LSTM layer
        z = h_n[-1]

        # Forecast future values
        pred = self.fc(z)

        if return_latents:
            return pred, z, z

        return pred

In [21]:
def create_windows(data, seq_len, horizon):
    X, y = [], []

    for i in range(len(data) - seq_len - horizon):

        # p historical timestamps
        x_window = data[i:i + seq_len]

        # H future timestamps for each channel
        y_window = data[
            i + seq_len:i + seq_len + horizon
        ].T.reshape(-1)

        X.append(x_window)
        y.append(y_window)

    X = torch.stack(X).float()
    y = torch.stack(y).float()

    return X, y


# Configuration for testing
p = 40
H = 16

X_train, y_train = create_windows(train_data, p, H)
X_test, y_test = create_windows(test_data, p, H)

print("Patch Length:", p)
print("Forecast Horizon:", H)

print("\nTrain:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTest:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nFirst target shape:", y_train[0].shape)
print("Contains NaN:", torch.isnan(X_train).any().item())

Patch Length: 40
Forecast Horizon: 16

Train:
X_train: torch.Size([2264, 40, 2])
y_train: torch.Size([2264, 32])

Test:
X_test: torch.Size([524, 40, 2])
y_test: torch.Size([524, 32])

First target shape: torch.Size([32])
Contains NaN: False


In [22]:
def compute_loss(
    pred,
    target,
    z,
    z_next,
    lyap_weight=0.1
):
    # 1. Prediction loss
    mse_loss = nn.functional.mse_loss(pred, target)

    # 2. Lyapunov regularization
    if lyap_weight > 0:

        # P = Identity matrix
        P = torch.eye(
            z.shape[1],
            device=z.device,
            dtype=z.dtype
        )

        # V(z) = z^T P z
        V_z = torch.einsum("bi,ij,bj->b", z, P, z)

        # V(z_next) = z_next^T P z_next
        V_z_next = torch.einsum(
            "bi,ij,bj->b",
            z_next, P, z_next
        )

        # Penalize positive increase
        lyap_loss = torch.relu(
            V_z_next - V_z
        ).mean()

    else:
        lyap_loss = torch.zeros_like(mse_loss)

    # 3. Total loss
    total_loss = mse_loss + lyap_weight * lyap_loss

    return total_loss, mse_loss, lyap_loss

In [24]:
def train_model(
    model,
    X_train,
    y_train,
    epochs=4000,
    lr=3e-4,
    lyap_weight=0.1,
    verbose=True
):
    model = model.to(DEVICE)

    X_train = X_train.to(DEVICE)
    y_train = y_train.to(DEVICE)

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr
    )

    # Lưu lịch sử huấn luyện
    loss_history = []
    mse_history = []
    lyap_history = []
    eig_history = []

    log_interval = max(1, epochs // 10)

    for epoch in range(epochs):

        model.train()

        # 1. Reset gradients
        optimizer.zero_grad()

        # 2. Forward pass
        pred, z, z_next = model(
            X_train,
            return_latents=True
        )

        # 3. Compute loss
        total_loss, mse_loss, lyap_loss = compute_loss(
            pred=pred,
            target=y_train,
            z=z,
            z_next=z_next,
            lyap_weight=lyap_weight
        )

        # 4. Backpropagation
        total_loss.backward()

        # 5. Update parameters
        optimizer.step()

        # 6. Save training history
        loss_history.append(total_loss.item())
        mse_history.append(mse_loss.item())
        lyap_history.append(lyap_loss.item())

        # Theo dõi đại lượng giống script tác giả:
        # max(sigmoid(S_raw)), không phải eigenvalue thực của K
        if hasattr(model, "koop"):
            with torch.no_grad():
                eig_max = (
                    torch.sigmoid(model.koop.S_raw)
                    .abs()
                    .max()
                    .item()
                )
        else:
            eig_max = 0.0

        eig_history.append(eig_max)

        if verbose and (
            epoch % log_interval == 0
            or epoch == epochs - 1
        ):
            print(
                f"Epoch {epoch+1:4d}/{epochs} | "
                f"MSE: {mse_loss.item():.6f} | "
                f"Lyap: {lyap_loss.item():.6f} | "
                f"Total: {total_loss.item():.6f}"
            )

    return {
        "model": model,
        "loss": np.array(loss_history),
        "mse": np.array(mse_history),
        "lyap": np.array(lyap_history),
        "eig": np.array(eig_history)
    }

In [26]:
def evaluate_model(model, X, y, batch_size=256):
    model.eval()

    device = next(model.parameters()).device

    all_predictions = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            x_batch = X[i:i + batch_size].to(device)

            pred = model(x_batch)

            all_predictions.append(pred.cpu())

    predictions = torch.cat(all_predictions, dim=0)

    # Calculate evaluation metrics
    mse = torch.mean((predictions - y) ** 2).item()
    mae = torch.mean(torch.abs(predictions - y)).item()

    return {
        "MSE": mse,
        "MAE": mae,
        "predictions": predictions
    }


# Đánh giá mô hình vừa smoke test 10 epochs
trained_model = result["model"]

train_metrics = evaluate_model(
    trained_model,
    X_train,
    y_train
)

test_metrics = evaluate_model(
    trained_model,
    X_test,
    y_test
)

print("TRAIN:")
print("MSE:", train_metrics["MSE"])
print("MAE:", train_metrics["MAE"])

print("\nTEST:")
print("MSE:", test_metrics["MSE"])
print("MAE:", test_metrics["MAE"])

print("\nPrediction shape:",
      test_metrics["predictions"].shape)

TRAIN:
MSE: 0.14245085418224335
MAE: 0.2977956533432007

TEST:
MSE: 0.28838154673576355
MAE: 0.4402484893798828

Prediction shape: torch.Size([524, 32])
